In [2]:
# -*- coding: utf-8 -*-
"""PHAN 3: FEATURE ENGINEERING HOAN CHINH
- Chia du lieu NGAY DAU (truoc moi xu ly)
- Tao dac trung moi (login_failure_ratio, ip_risk_score, packet_density, session_age_group)
- Xu ly outlier, log transform, scaling (chi fit tren train)
- One-hot encoding va label encoding cho bien phan loai
- Luu preprocessor components va cac tap da xu ly
"""

import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

print("="*80)
print("PHAN 3: FEATURE ENGINEERING HOAN CHINH")
print("="*80)

# =============================================================================
# 1. DOC DU LIEU DA LAM SACH (TU PHAN 1)
# =============================================================================

print("\n1. DOC DU LIEU DA LAM SACH")
print("-"*50)

df = pd.read_parquet("../data_processed/cleaned_data.parquet")
print(f"Kich thuoc: {df.shape[0]} hang, {df.shape[1]} cot")
print(f"Ty le tan cong: {df['attack_detected'].mean():.2%}")

df_original = df.copy()

# =============================================================================
# 2. CHIA DU LIEU NGAY LAP TUC (TRUOC MOI XU LY)
# =============================================================================

print("\n" + "="*80)
print("2. CHIA DU LIEU TRAIN/VALIDATION/TEST (80/10/10)")
print("="*80)

target_col = 'attack_detected'
feature_cols_raw = [col for col in df.columns if col != target_col]

X_raw = df[feature_cols_raw].copy()
y_raw = df[target_col].copy()

X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"\nKich thuoc cac tap (RAW - truoc xu ly):")
print(f"   - Train: {X_train_raw.shape[0]} mau ({y_train.mean():.2%} attack)")
print(f"   - Validation: {X_val_raw.shape[0]} mau ({y_val.mean():.2%} attack)")
print(f"   - Test: {X_test_raw.shape[0]} mau ({y_test.mean():.2%} attack)")

# =============================================================================
# 3. TAO DAC TRUNG MOI (FEATURE CREATION) - TREN TUNG TAP RIENG
# =============================================================================

print("\n" + "="*80)
print("3. TAO DAC TRUNG MOI (FEATURE CREATION)")
print("="*80)

def create_features(df_input):
    df = df_input.copy()
    
    # 3.1. Ty le dang nhap that bai (login_failure_ratio)
    df['login_failure_ratio'] = df['failed_logins'] / df['login_attempts']
    df['login_failure_ratio'] = df['login_failure_ratio'].fillna(0)
    df['login_failure_ratio'] = df['login_failure_ratio'].clip(0, 1)
    
    # 3.2. Diem rui ro IP (ip_risk_score)
    df['ip_risk_score'] = df['ip_reputation_score'] * (df['unusual_time_access'] + 1)
    
    # 3.3. Mat do goi tin (packet_density)
    df['packet_density'] = df['network_packet_size'] / (df['session_duration'] + 1e-6)
    
    # 3.4. Nhom thoi luong phien (session_age_group)
    def classify_session_duration(duration):
        if duration < 60:
            return 'very_short'
        elif duration < 300:
            return 'short'
        elif duration < 3600:
            return 'medium'
        elif duration < 86400:
            return 'long'
        else:
            return 'very_long'
    
    df['session_age_group'] = df['session_duration'].apply(classify_session_duration)
    
    return df

print("\nDang tao dac trung moi cho tung tap...")

X_train_raw = create_features(X_train_raw)
X_val_raw = create_features(X_val_raw)
X_test_raw = create_features(X_test_raw)

new_features = ['login_failure_ratio', 'ip_risk_score', 'packet_density', 'session_age_group']
print(f"Da tao {len(new_features)} dac trung moi: {new_features}")

print("\nThong ke cac dac trung moi (tren tap train):")
for col in new_features:
    if col != 'session_age_group':
        print(f"   - {col}: mean={X_train_raw[col].mean():.4f}, std={X_train_raw[col].std():.4f}")
    else:
        print(f"   - {col}: {X_train_raw[col].value_counts().to_dict()}")

# =============================================================================
# 4. PHAN LOAI DAC TRUNG (CHI DUA TREN X_TRAIN)
# =============================================================================

print("\n" + "="*80)
print("4. PHAN LOAI DAC TRUNG")
print("="*80)

feature_types = {
    'ID features': [],
    'Numerical features': [],
    'Categorical features': [],
    'Binary features': [],
    'Target feature': []
}

for col in X_train_raw.columns:
    if col == 'session_id':
        feature_types['ID features'].append(col)
    elif X_train_raw[col].dtype in ['int64', 'float64']:
        unique_vals = X_train_raw[col].nunique()
        if unique_vals == 2 and set(X_train_raw[col].unique()).issubset({0, 1}):
            feature_types['Binary features'].append(col)
        else:
            feature_types['Numerical features'].append(col)
    else:
        feature_types['Categorical features'].append(col)

print("\nPHAN LOAI DAC TRUNG:")
for ftype, cols in feature_types.items():
    if cols:
        print(f"\n   {ftype}: {len(cols)} cot")
        print(f"     -> {cols}")

numeric_cols = feature_types['Numerical features'].copy()
categorical_cols = feature_types['Categorical features'].copy()
binary_cols = feature_types['Binary features'].copy()

if feature_types['ID features']:
    print(f"\nDa bo qua cot ID: {feature_types['ID features']}")

print(f"\nSo luong dac trung so: {len(numeric_cols)}")
print(f"So luong dac trung phan loai: {len(categorical_cols)}")
print(f"So luong dac trung nhi phan: {len(binary_cols)}")

# =============================================================================
# 5. THONG KE MO TA TRUOC KHI XU LY
# =============================================================================

print("\n" + "="*80)
print("5. THONG KE MO TA TRUOC KHI XU LY")
print("="*80)

summary_before = []
for col in numeric_cols:
    summary_before.append({
        'Feature': col,
        'Min': round(X_train_raw[col].min(), 2),
        'Q1': round(X_train_raw[col].quantile(0.25), 2),
        'Median': round(X_train_raw[col].median(), 2),
        'Q3': round(X_train_raw[col].quantile(0.75), 2),
        'Max': round(X_train_raw[col].max(), 2),
        'Mean': round(X_train_raw[col].mean(), 2),
        'Std': round(X_train_raw[col].std(), 2),
        'Skewness': round(X_train_raw[col].skew(), 4)
    })

summary_before_df = pd.DataFrame(summary_before)
print("\nBang thong ke mo ta (tren tap train):")
print(summary_before_df.to_string(index=False))

# =============================================================================
# 6. DINH NGHIA CAC TRANSFORMER TUY CHINH
# =============================================================================

print("\n" + "="*80)
print("6. DINH NGHIA CAC TRANSFORMER")
print("="*80)

class OutlierClipper(BaseEstimator, TransformerMixin):
    def __init__(self, factor=1.5):
        self.factor = factor
        self.lower_bounds_ = {}
        self.upper_bounds_ = {}
    
    def fit(self, X, y=None):
        for col in X.columns:
            Q1 = X[col].quantile(0.25)
            Q3 = X[col].quantile(0.75)
            IQR = Q3 - Q1
            self.lower_bounds_[col] = Q1 - self.factor * IQR
            self.upper_bounds_[col] = Q3 + self.factor * IQR
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        for col in X.columns:
            X_copy[col] = X_copy[col].clip(self.lower_bounds_[col], self.upper_bounds_[col])
        return X_copy

class LogTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, skew_threshold=1.0):
        self.skew_threshold = skew_threshold
        self.log_cols_ = []
    
    def fit(self, X, y=None):
        for col in X.columns:
            if X[col].skew() > self.skew_threshold:
                self.log_cols_.append(col)
        return self
    
    def transform(self, X):
        X_copy = X.copy()
        for col in self.log_cols_:
            X_copy[f'{col}_log'] = np.log1p(X_copy[col])
            X_copy = X_copy.drop(columns=[col])
        return X_copy

class CustomScaler(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.scaler = StandardScaler()
        self.feature_names_in_ = None
    
    def fit(self, X, y=None):
        self.scaler.fit(X)
        self.feature_names_in_ = X.columns if hasattr(X, 'columns') else None
        return self
    
    def transform(self, X):
        scaled = self.scaler.transform(X)
        if self.feature_names_in_ is not None:
            return pd.DataFrame(scaled, columns=self.feature_names_in_)
        return scaled

# =============================================================================
# 7. XU LY DAC TRUNG SO (Outlier + Log + Scaling)
# =============================================================================

print("\n" + "="*80)
print("7. XU LY DAC TRUNG SO")
print("="*80)

print("\n7.1. XU LY OUTLIER (IQR method)...")
outlier_clipper = OutlierClipper(factor=1.5)
outlier_clipper.fit(X_train_raw[numeric_cols])

outlier_report = []
for col in numeric_cols:
    lower = outlier_clipper.lower_bounds_[col]
    upper = outlier_clipper.upper_bounds_[col]
    Q1 = X_train_raw[col].quantile(0.25)
    Q3 = X_train_raw[col].quantile(0.75)
    IQR = Q3 - Q1
    
    outliers_before = ((X_train_raw[col] < lower) | (X_train_raw[col] > upper)).sum()
    outlier_pct = (outliers_before / len(X_train_raw)) * 100
    
    outlier_report.append({
        'Feature': col,
        'Q1': round(Q1, 2),
        'Q3': round(Q3, 2),
        'IQR': round(IQR, 2),
        'Lower bound': round(lower, 2),
        'Upper bound': round(upper, 2),
        'Outliers (%)': f"{outlier_pct:.2f}%"
    })

outlier_report_df = pd.DataFrame(outlier_report)
print("\nBAO CAO OUTLIER:")
print(outlier_report_df.to_string(index=False))

X_train_num = outlier_clipper.transform(X_train_raw[numeric_cols])
X_val_num = outlier_clipper.transform(X_val_raw[numeric_cols])
X_test_num = outlier_clipper.transform(X_test_raw[numeric_cols])
print("Da xu ly outlier cho ca 3 tap")

print("\n7.2. LOG TRANSFORM (cho cot lech > 1)...")
log_transformer = LogTransformer(skew_threshold=1.0)
log_transformer.fit(X_train_num)

print(f"\n   Cac cot duoc log transform: {log_transformer.log_cols_}")

X_train_log = log_transformer.transform(X_train_num)
X_val_log = log_transformer.transform(X_val_num)
X_test_log = log_transformer.transform(X_test_num)

print("\n7.3. SCALING (StandardScaler)...")
scaler = CustomScaler()
X_train_scaled = scaler.fit_transform(X_train_log)
X_val_scaled = scaler.transform(X_val_log)
X_test_scaled = scaler.transform(X_test_log)

print(f"Da scaling {X_train_scaled.shape[1]} dac trung so")

# =============================================================================
# 8. XU LY DAC TRUNG PHAN LOAI
# =============================================================================

print("\n" + "="*80)
print("8. XU LY DAC TRUNG PHAN LOAI")
print("="*80)

print(f"\nCac dac trung phan loai: {categorical_cols}")

print("\n8.1. PHAN TICH CHI TIET TUNG COT PHAN LOAI:")
for col in categorical_cols:
    unique_vals = X_train_raw[col].nunique()
    print(f"\n   {col}:")
    print(f"     - So gia tri duy nhat: {unique_vals}")
    print(f"     - Phan phoi:")
    print(X_train_raw[col].value_counts().head(5).to_string())
    if 'Unknown' in X_train_raw[col].values:
        unknown_pct = (X_train_raw[col] == 'Unknown').mean() * 100
        print(f"     - Gia tri 'Unknown' chiem: {unknown_pct:.2f}%")

print("\n8.2. MA HOA CAC DAC TRUNG PHAN LOAI:")
print("-"*50)

one_hot_cols = []
label_encode_cols = []
label_encoders = {}

for col in categorical_cols:
    n_unique = X_train_raw[col].nunique()
    
    if n_unique <= 5:
        one_hot_cols.append(col)
        print(f"\n   {col}: One-Hot Encoding (co {n_unique} gia tri)")
    else:
        label_encode_cols.append(col)
        print(f"\n   {col}: Label Encoding (co {n_unique} gia tri)")

if one_hot_cols:
    print("\n   Thuc hien One-Hot Encoding...")
    encoder = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
    encoder.fit(X_train_raw[one_hot_cols])
    
    X_train_cat_onehot = encoder.transform(X_train_raw[one_hot_cols])
    X_val_cat_onehot = encoder.transform(X_val_raw[one_hot_cols])
    X_test_cat_onehot = encoder.transform(X_test_raw[one_hot_cols])
    
    onehot_feature_names = []
    for i, col in enumerate(one_hot_cols):
        cats = encoder.categories_[i]
        for cat in cats[1:]:
            onehot_feature_names.append(f"{col}_{cat}")
    
    print(f"     -> Da tao {len(onehot_feature_names)} cot one-hot")
else:
    X_train_cat_onehot = np.empty((len(X_train_raw), 0))
    X_val_cat_onehot = np.empty((len(X_val_raw), 0))
    X_test_cat_onehot = np.empty((len(X_test_raw), 0))
    onehot_feature_names = []

if label_encode_cols:
    print("\n   Thuc hien Label Encoding...")
    label_encoded_arrays = []
    
    for col in label_encode_cols:
        le = LabelEncoder()
        train_encoded = le.fit_transform(X_train_raw[col])
        val_encoded = le.transform(X_val_raw[col])
        test_encoded = le.transform(X_test_raw[col])
        
        label_encoders[col] = le
        
        label_encoded_arrays.append((
            train_encoded.reshape(-1, 1),
            val_encoded.reshape(-1, 1),
            test_encoded.reshape(-1, 1),
            col
        ))
        
        mapping = dict(zip(le.classes_, le.transform(le.classes_)))
        print(f"     -> {col}: mapping mau {dict(list(mapping.items())[:3])}")
    
    if label_encoded_arrays:
        X_train_label = np.hstack([arr[0] for arr in label_encoded_arrays])
        X_val_label = np.hstack([arr[1] for arr in label_encoded_arrays])
        X_test_label = np.hstack([arr[2] for arr in label_encoded_arrays])
        label_feature_names = [arr[3] for arr in label_encoded_arrays]
    else:
        X_train_label = np.empty((len(X_train_raw), 0))
        X_val_label = np.empty((len(X_val_raw), 0))
        X_test_label = np.empty((len(X_test_raw), 0))
        label_feature_names = []
else:
    X_train_label = np.empty((len(X_train_raw), 0))
    X_val_label = np.empty((len(X_val_raw), 0))
    X_test_label = np.empty((len(X_test_raw), 0))
    label_feature_names = []

# =============================================================================
# 9. XU LY DAC TRUNG NHI PHAN
# =============================================================================

print("\n" + "="*80)
print("9. XU LY DAC TRUNG NHI PHAN")
print("="*80)

print(f"Cac dac trung nhi phan: {binary_cols}")

if binary_cols:
    for col in binary_cols:
        X_train_raw[col] = X_train_raw[col].astype(int)
        X_val_raw[col] = X_val_raw[col].astype(int)
        X_test_raw[col] = X_test_raw[col].astype(int)
    
    X_train_binary = X_train_raw[binary_cols].values
    X_val_binary = X_val_raw[binary_cols].values
    X_test_binary = X_test_raw[binary_cols].values
    binary_feature_names = binary_cols
else:
    X_train_binary = np.empty((len(X_train_raw), 0))
    X_val_binary = np.empty((len(X_val_raw), 0))
    X_test_binary = np.empty((len(X_test_raw), 0))
    binary_feature_names = []

# =============================================================================
# 10. GHEP TAT CA CAC DAC TRUNG
# =============================================================================

print("\n" + "="*80)
print("10. GHEP TAT CA CAC DAC TRUNG")
print("="*80)

X_train_processed = np.hstack([X_train_scaled, X_train_cat_onehot, X_train_label, X_train_binary])
X_val_processed = np.hstack([X_val_scaled, X_val_cat_onehot, X_val_label, X_val_binary])
X_test_processed = np.hstack([X_test_scaled, X_test_cat_onehot, X_test_label, X_test_binary])

all_feature_names = (
    list(X_train_log.columns) +
    onehot_feature_names +
    label_feature_names +
    binary_feature_names
)

print(f"Tong so dac trung sau xu ly: {len(all_feature_names)}")
print(f"   - Numerical (scaled): {X_train_scaled.shape[1]}")
print(f"   - One-hot: {len(onehot_feature_names)}")
print(f"   - Label encoded: {len(label_feature_names)}")
print(f"   - Binary: {len(binary_feature_names)}")

# =============================================================================
# 11. LUU DU LIEU DA XU LY
# =============================================================================

print("\n" + "="*80)
print("11. LUU DU LIEU DA XU LY")
print("="*80)

os.makedirs("../data_processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

joblib.dump(X_train_processed, "../data_processed/X_train_processed.pkl")
joblib.dump(y_train, "../data_processed/y_train.pkl")
joblib.dump(X_val_processed, "../data_processed/X_val_processed.pkl")
joblib.dump(y_val, "../data_processed/y_val.pkl")
joblib.dump(X_test_processed, "../data_processed/X_test_processed.pkl")
joblib.dump(y_test, "../data_processed/y_test.pkl")

preprocessor_components = {
    'outlier_clipper': outlier_clipper,
    'log_transformer': log_transformer,
    'scaler': scaler,
    'onehot_encoder': encoder if one_hot_cols else None,
    'label_encoders': label_encoders,
    'feature_names': all_feature_names,
    'numeric_cols': numeric_cols,
    'categorical_cols': categorical_cols,
    'binary_cols': binary_cols,
    'one_hot_cols': one_hot_cols,
    'label_encode_cols': label_encode_cols
}

joblib.dump(preprocessor_components, "../models/preprocessor_components.pkl")
joblib.dump(all_feature_names, "../models/feature_names.pkl")
X_test_raw.to_parquet("../data_processed/X_test_raw.parquet", index=False)

print("\nDa luu:")
print("   data_processed/X_train_processed.pkl")
print("   data_processed/X_val_processed.pkl")
print("   data_processed/X_test_processed.pkl")
print("   data_processed/y_train.pkl, y_val.pkl, y_test.pkl")
print("   data_processed/X_test_raw.parquet")
print("   models/preprocessor_components.pkl")
print("   models/feature_names.pkl")

# =============================================================================
# 12. KIEM TRA NHANH
# =============================================================================

print("\n" + "="*80)
print("12. KIEM TRA NHANH KET QUA")
print("="*80)

print(f"\nThong ke X_train_processed (sau scaling):")
print(f"   - Mean: {X_train_processed.mean():.4f}")
print(f"   - Std:  {X_train_processed.std():.4f}")
print(f"   - Min:  {X_train_processed.min():.4f}")
print(f"   - Max:  {X_train_processed.max():.4f}")

print(f"\nThong ke y_train:")
print(f"   - Normal (0): {(y_train == 0).sum():,} mau")
print(f"   - Attack (1): {(y_train == 1).sum():,} mau")
print(f"   - Ty le tan cong: {y_train.mean():.2%}")

print(f"\nKIEM TRA RO RI DU LIEU:")
print(f"   - Fit cac transformers tren: Train ({X_train_raw.shape[0]} mau)")
print(f"   - Transform validation: {X_val_processed.shape[0]} mau (doc lap)")
print(f"   - Transform test: {X_test_processed.shape[0]} mau (doc lap)")
print(f"   -> KHONG co ro ri du lieu tu validation/test vao train!")

# =============================================================================
# 13. TOM TAT
# =============================================================================

print("\n" + "="*80)
print("13. TOM TAT QUY TRINH XU LY DAC TRUNG")
print("="*80)

print(f"""
TOM TAT QUY TRINH XU LY DAC TRUNG HOAN CHINH
================================================

1. CHIA DU LIEU:
   - Train: {X_train_raw.shape[0]} mau ({y_train.mean():.2%} attack)
   - Validation: {X_val_raw.shape[0]} mau ({y_val.mean():.2%} attack)
   - Test: {X_test_raw.shape[0]} mau ({y_test.mean():.2%} attack)
   -> Chia NGAY DAU, truoc moi xu ly

2. TAO DAC TRUNG MOI:
   - login_failure_ratio (ty le dang nhap that bai)
   - ip_risk_score (diem rui ro IP)
   - packet_density (mat do goi tin)
   - session_age_group (phan nhom thoi luong phien)

3. XU LY DAC TRUNG SO:
   - So luong: {len(numeric_cols)} cot
   - Xu ly outlier (IQR, factor=1.5): {len(outlier_report)} cot
   - Log transform (skew > 1): {len(log_transformer.log_cols_)} cot
   - Scaling (StandardScaler): {X_train_scaled.shape[1]} cot

4. XU LY DAC TRUNG PHAN LOAI:
   - So luong: {len(categorical_cols)} cot
   - One-Hot Encoding: {len(one_hot_cols)} cot -> {len(onehot_feature_names)} cot
   - Label Encoding: {len(label_encode_cols)} cot -> {len(label_feature_names)} cot

5. XU LY DAC TRUNG NHI PHAN:
   - So luong: {len(binary_cols)} cot -> giu nguyen

6. KET QUA:
   - Du lieu dau vao: {df_original.shape[0]} hang, {df_original.shape[1]} cot
   - Du lieu dau ra: {X_train_processed.shape[0]} hang, {X_train_processed.shape[1]} cot
   -> KHONG ro ri du lieu
   -> San sang cho huan luyen mo hinh
""")

print("\nHOAN THANH FEATURE ENGINEERING (KHONG RO RI)")
print("="*80)

PHAN 3: FEATURE ENGINEERING HOAN CHINH

1. DOC DU LIEU DA LAM SACH
--------------------------------------------------
Kich thuoc: 7571 hang, 10 cot
Ty le tan cong: 44.30%

2. CHIA DU LIEU TRAIN/VALIDATION/TEST (80/10/10)

Kich thuoc cac tap (RAW - truoc xu ly):
   - Train: 6056 mau (44.30% attack)
   - Validation: 757 mau (44.25% attack)
   - Test: 758 mau (44.33% attack)

3. TAO DAC TRUNG MOI (FEATURE CREATION)

Dang tao dac trung moi cho tung tap...
Da tao 4 dac trung moi: ['login_failure_ratio', 'ip_risk_score', 'packet_density', 'session_age_group']

Thong ke cac dac trung moi (tren tap train):
   - login_failure_ratio: mean=0.4426, std=0.3354
   - ip_risk_score: mean=0.3820, std=0.2434
   - packet_density: mean=5.1957, std=44.3839
   - session_age_group: {'medium': 4101, 'short': 1445, 'very_short': 438, 'long': 72}

4. PHAN LOAI DAC TRUNG

PHAN LOAI DAC TRUNG:

   Numerical features: 8 cot
     -> ['network_packet_size', 'login_attempts', 'session_duration', 'ip_reputation_score'